# **OpenRocket landing positions**
Plots the landing positions of an OpenRocket parameter sweep using the config from the corresponding project.

The sweep notebook (`openrocket_parameter_sweep.ipynb` in the OpenRocket repository) writes `openrocket_landing_positions.csv`
into the project folder. The landing points are placed relative to the launch rail in `config.toml`.

Set the **Configuration** below, then *Run All*. The plot is also saved as `plots/<PLOT_NAME>.html` in the project folder.

## Configuration

In [ ]:
PROJECT = "ALBATROSS_II"

## Imports

In [ ]:
import os
from pathlib import Path

os.chdir(next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "simulation_core").is_dir()))

import numpy as np
import pandas as pd

from simulation_core.outputs import (
    SCENARIO_COLORS, LandingPoint, classify_headings, plot_landing_positions_with_modes, print_safety_summary, print_stats_table,
)
from simulation_core.utils import latlon_to_local_xy, load_config, load_zones, scale_zones

CONFIG_PATH = Path(f"projects/{PROJECT}/config.toml")
ZONES_PATH = Path(f"projects/{PROJECT}/zones.toml")
LANDING_POSITIONS_PATH = Path(f"projects/{PROJECT}/openrocket_landing_positions.csv")   # written by openrocket_parameter_sweep.ipynb

# CSV input columns of the sweep notebook that set the wind; they form the Environment column of the stats tables.
WIND_PROFILE_COLUMN = "wind profile"
AVERAGE_WIND_COLUMN = "average wind speed [m/s]"

## Read landing positions

In [ ]:
def read_landing_points(landing_table: pd.DataFrame, input_columns: list[str], rail_latitude: float, rail_longitude: float) -> list[LandingPoint]:
    """Convert the OpenRocket landing positions (lat/lon) into LandingPoints, in meters east/north of the launch rail from config.toml."""
    landing_rows = landing_table.to_dict("records")

    launch_offsets_m = [
        np.hypot(*latlon_to_local_xy(row["launch_latitude"], row["launch_longitude"], rail_latitude, rail_longitude))
        for row in landing_rows
    ]
    if max(launch_offsets_m) > 1:
        print(f"Warning: the OpenRocket launch site is up to {max(launch_offsets_m):.0f} m away from the launch rail in config.toml. "
              "Set the same coordinates in the .ork simulation options.")

    landing_points = []
    for row in landing_rows:
        x_east_m, y_north_m = latlon_to_local_xy(row["landing_latitude"], row["landing_longitude"], rail_latitude, rail_longitude)
        landing_points.append(LandingPoint(
            scenario=row["scenario"],
            heading=row["heading"],
            inclination=row["inclination"],
            x_east_m=x_east_m,
            y_north_m=y_north_m,
            latitude=row["landing_latitude"],
            longitude=row["landing_longitude"],
            configuration={column: f"{row[column]:g}" if isinstance(row[column], float) else str(row[column]) for column in input_columns},
        ))
    return landing_points


def print_openrocket_stats(landing_table: pd.DataFrame, input_columns: list[str], metric_columns: list[str]):
    """Print one "Stats for <scenario> flights" table per scenario"""
    # Like RocketPy's Config (flight._meta), only values that differ between the runs; heading and inclination have own CSV columns.
    config_candidates = [column for column in input_columns if column not in (WIND_PROFILE_COLUMN, AVERAGE_WIND_COLUMN)] + ["heading", "inclination"]
    varied_columns = [column for column in config_candidates if landing_table[column].nunique() > 1]

    for scenario in [scenario for scenario in SCENARIO_COLORS if scenario in set(landing_table["scenario"])]:
        rows = []
        for row in landing_table[landing_table["scenario"] == scenario].to_dict("records"):
            if WIND_PROFILE_COLUMN in row:
                environment = row[WIND_PROFILE_COLUMN]
            elif AVERAGE_WIND_COLUMN in row:
                environment = f"{row[AVERAGE_WIND_COLUMN]:g}_m_s_AVERAGE_WIND"
            else:
                environment = ".ork wind"       # wind from .ork file
            config = ",\n".join(
                f"{column}={row[column]:g}" if isinstance(row[column], float) else f"{column}={row[column]}" for column in varied_columns
            )
            # Empty cells are metrics the scenario does not have (no_main, ballistic), so these columns are left out.
            metrics = {key: row[key] for key in metric_columns if pd.notna(row[key])}
            rows.append({"Environment": environment, "Config": config or "-", **metrics})
        print_stats_table(scenario, rows)

## Plot

In [ ]:
params = load_config(CONFIG_PATH)
exclusion_zones, buffer_zones, suboptimal_zones, exclusion_zone_safety_margin, satellite_image = load_zones(ZONES_PATH)
buffer_zones.update(scale_zones(exclusion_zones, exclusion_zone_safety_margin))

landing_table = pd.read_csv(LANDING_POSITIONS_PATH)
columns = list(landing_table.columns)
input_columns = columns[:columns.index("scenario")]
metric_columns = columns[columns.index("landing_longitude") + 1:]
numeric_metrics = landing_table[metric_columns].apply(pd.to_numeric, errors="coerce")
landing_table[metric_columns] = numeric_metrics.where(numeric_metrics.notna(), landing_table[metric_columns])

landing_points = read_landing_points(landing_table, input_columns, params.config.environment.latitude, params.config.environment.longitude)
label_by_heading = classify_headings(landing_points, buffer_zones, suboptimal_zones)

print("INFO: Inclination is using the RocketPy convention: 0° is horizontal, 90° is vertical!")
print_openrocket_stats(landing_table, input_columns, metric_columns)
print_safety_summary(landing_points, label_by_heading, buffer_zones, suboptimal_zones)

plot_landing_positions_with_modes(
    params,
    exclusion_zones,
    buffer_zones,
    suboptimal_zones,
    plot_name="openrocket_landing_positions",
    landing_points=landing_points,
    label_by_heading=label_by_heading,
    satellite_image=satellite_image,
    source="OpenRocket"
)